In [ ]:
# Install dependencies
!pip install -q sentence-transformers

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:
import os

# Update this path to wherever your files are in Drive
base_path = '/content/drive/MyDrive'  # change if in a subfolder


In [ ]:
import os
import json
import re

# ─────────────────────────────────────────
# CONFIGURE THESE PATHS FOR YOUR DRIVE
# ─────────────────────────────────────────
NCD_FOLDER = '/content/drive/MyDrive/GenAI_project/Data/ncd_docs'
LCD_FOLDER = '/content/drive/MyDrive/GenAI_project/Data/lcd_docs_v2/lcd_docs_v2'

OUTPUT_FILE = '/content/drive/MyDrive/GenAI_project/Data/all_chunks.json'

CHUNK_SIZE  = 400   # words per chunk
OVERLAP     = 50    # word overlap between chunks

# ─────────────────────────────────────────
# CHUNKING FUNCTION
# ─────────────────────────────────────────
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=OVERLAP):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = ' '.join(words[i:i+chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
        if i + overlap >= len(words):
            break
    return chunks

# ─────────────────────────────────────────
# PARSE NCD METADATA FROM FILE HEADER
# ─────────────────────────────────────────
def parse_ncd_file(filepath, filename):
    with open(filepath, encoding='utf-8') as f:
        content = f.read()

    # Extract title from first line if it starts with TITLE:
    lines = content.split('\n')
    title = filename.replace('_', ' ').replace('.txt', '')
    ncd_id = 'UNKNOWN'

    for line in lines[:10]:
        if line.startswith('TITLE:'):
            title = line.replace('TITLE:', '').strip()
        if line.startswith('NCD_ID:') or line.startswith('SECTION:'):
            ncd_id = line.split(':', 1)[-1].strip()

    text_chunks = chunk_text(content)

    results = []
    for i, chunk in enumerate(text_chunks):
        # Prepend metadata so it's never lost even if chunk is shown in isolation
        tagged = f"[TYPE: NCD | NCD_ID: {ncd_id} | TITLE: {title}]\n\n{chunk}"
        results.append({
            'text':      tagged,
            'raw_text':  chunk,
            'source_id': ncd_id,
            'title':     title,
            'type':      'NCD',
            'states':    ['ALL'],   # NCDs apply nationally
            'filename':  filename,
            'chunk_idx': i
        })
    return results

# ─────────────────────────────────────────
# PARSE LCD METADATA FROM FILE HEADER
# ─────────────────────────────────────────
def parse_lcd_file(filepath, filename):
    with open(filepath, encoding='utf-8') as f:
        content = f.read()

    lines = content.split('\n')
    title      = filename.replace('_', ' ').replace('.txt', '')
    lcd_id     = 'UNKNOWN'
    contractor = 'UNKNOWN'
    states     = ['ALL']

    for line in lines[:15]:
        if line.startswith('TITLE:'):
            title = line.replace('TITLE:', '').strip()
        elif line.startswith('LCD_ID:'):
            lcd_id = line.replace('LCD_ID:', '').strip()
        elif line.startswith('CONTRACTOR:'):
            contractor = line.replace('CONTRACTOR:', '').strip()
        elif line.startswith('STATES:'):
            raw = line.replace('STATES:', '').strip()
            states = [s.strip() for s in raw.split(',') if s.strip()]

    text_chunks = chunk_text(content)

    results = []
    for i, chunk in enumerate(text_chunks):
        tagged = f"[TYPE: LCD | LCD_ID: {lcd_id} | TITLE: {title} | STATES: {', '.join(states)} | CONTRACTOR: {contractor}]\n\n{chunk}"
        results.append({
            'text':       tagged,
            'raw_text':   chunk,
            'source_id':  lcd_id,
            'title':      title,
            'type':       'LCD',
            'states':     states,
            'contractor': contractor,
            'filename':   filename,
            'chunk_idx':  i
        })
    return results

# ─────────────────────────────────────────
# PROCESS ALL FILES
# ─────────────────────────────────────────
all_chunks = []
ncd_count  = 0
lcd_count  = 0
skipped    = 0

# Process NCDs
print("Processing NCD files...")
for fname in sorted(os.listdir(NCD_FOLDER)):
    if not fname.endswith('.txt'):
        continue
    fpath = os.path.join(NCD_FOLDER, fname)
    try:
        chunks = parse_ncd_file(fpath, fname)
        all_chunks.extend(chunks)
        ncd_count += 1
    except Exception as e:
        print(f"  Skipped {fname}: {e}")
        skipped += 1

print(f"  ✓ {ncd_count} NCD files → {sum(1 for c in all_chunks if c['type']=='NCD')} chunks")

# Process LCDs
print("Processing LCD files...")
lcd_start = len(all_chunks)
for fname in sorted(os.listdir(LCD_FOLDER)):
    if not fname.endswith('.txt'):
        continue
    fpath = os.path.join(LCD_FOLDER, fname)
    try:
        chunks = parse_lcd_file(fpath, fname)
        all_chunks.extend(chunks)
        lcd_count += 1
    except Exception as e:
        print(f"  Skipped {fname}: {e}")
        skipped += 1

print(f"  ✓ {lcd_count} LCD files → {sum(1 for c in all_chunks if c['type']=='LCD')} chunks")

# ─────────────────────────────────────────
# SAVE OUTPUT
# ─────────────────────────────────────────
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(all_chunks, f, indent=2, ensure_ascii=False)

print(f"\n{'='*50}")
print(f"DONE!")
print(f"  Total chunks:  {len(all_chunks)}")
print(f"  NCD files:     {ncd_count}")
print(f"  LCD files:     {lcd_count}")
print(f"  Skipped:       {skipped}")
print(f"  Output saved:  {OUTPUT_FILE}")
print(f"{'='*50}")

Processing NCD files...
  ✓ 343 NCD files → 575 chunks
Processing LCD files...
  ✓ 819 LCD files → 2802 chunks

DONE!
  Total chunks:  3377
  NCD files:     343
  LCD files:     819
  Skipped:       0
  Output saved:  /content/drive/MyDrive/GenAI_project/Data/all_chunks.json


In [ ]:
import json

chunks = json.load(open('/content/drive/MyDrive/GenAI_project/Data/all_chunks.json'))

print(f"Total chunks: {len(chunks)}")
print(f"NCD chunks:   {sum(1 for c in chunks if c['type'] == 'NCD')}")
print(f"LCD chunks:   {sum(1 for c in chunks if c['type'] == 'LCD')}")

# Show one NCD chunk
ncd_sample = next(c for c in chunks if c['type'] == 'NCD')
print("\n--- NCD SAMPLE ---")
print(ncd_sample['text'][:400])

# Show one Texas LCD chunk
tx_sample = next((c for c in chunks if 'TX' in c.get('states', [])), None)
print("\n--- TEXAS LCD SAMPLE ---")
print(tx_sample['text'][:400] if tx_sample else "No TX chunk found")

Total chunks: 3377
NCD chunks:   575
LCD chunks:   2802

--- NCD SAMPLE ---
[TYPE: NCD | NCD_ID: 100.3 | TITLE: 24-Hour Ambulatory Esophageal pH Monitoring]

TITLE: 24-Hour Ambulatory Esophageal pH Monitoring SECTION: 100.3 EFFECTIVE DATE: 1985-06-11 00:00:00 COVERAGE POLICY: Twenty-four hour ambulatory pH monitoring is covered by Medicare for patients who are suspected of having gastric reflux, but only if the patient presents diagnostic problems associated with atypical

--- TEXAS LCD SAMPLE ---
[TYPE: LCD | LCD_ID: 37792 | TITLE: 4Kscore Test Algorithm | STATES: AL, AK, AZ, AR, CA, CO, CT, DE, DC, FL, GA, HI, ID, IL, IN, IA, KS, KY, LA, ME, MD, MA, MI, MN, MS, MO, MT, NE, NV, NH, NJ, NM, NY, NC, ND, OH, OK, OR, PA, PR, RI, SC, SD, TN, TX, UT, VT, VA, VI, WA, WV, WI, WY | CONTRACTOR: NATIONAL]

TITLE: 4Kscore Test Algorithm LCD_ID: 37792 TYPE: Local Coverage Determination (LCD) CONTRACTOR
